# Step 1: 数据探索 — Sci-Plex2 (Srivatsan 2020)

本 notebook 只做一件事：**加载数据并弄清它的结构**。不做 PCA、UMAP 或任何下游分析。

数据来源：pertpy 内置的 `srivatsan_2020_sciplex2`，是论文 *Massively multiplex chemical transcriptomics at single-cell resolution*（Srivatsan et al., *Science* 2020）的一个子集：

- **1 种细胞系**：A549（人肺腺癌）
- **4 种药物** + 对照：地塞米松(Dex)、nutlin-3a、BMS-345541、vorinostat(SAHA)
- **多个剂量 × 多个重复**

> ⚠️ 命名提醒：pertpy 里还有 `srivatsan_2020_sciplex3`（188 药 × 65 万细胞的完整筛选，2.5GB）。本项目刻意用小的 sciplex2 作为教学数据。


## 先理解核心数据结构：AnnData

单细胞数据在 scanpy / pertpy 里都装在一个 `AnnData` 对象里。它有 6 个重要槽位（slot）：

| 槽位 | 存什么 | 形状 |
|---|---|---|
| `.X` | 表达矩阵（核心数据） | 细胞 × 基因 |
| `.obs` | **细胞**的 metadata（每一行 = 一个细胞） | 细胞 × 属性列 |
| `.var` | **基因**的 metadata（每一行 = 一个基因） | 基因 × 属性列 |
| `.layers` | 额外的表达矩阵（如归一化后、log 后） | 细胞 × 基因 |
| `.obsm` | 细胞的低维坐标（如 PCA、UMAP） | 细胞 × 维度 |
| `.uns` | 非结构化的元信息（颜色、参数、图等） | 字典 |

**两条铁律**：

1. `.obs` 和 `.X` 的**行**对齐 —— 都按「细胞」排。
2. `.var` 和 `.X` 的**列**对齐 —— 都按「基因」排。


## 先理解核心数据结构：AnnData

单细胞数据在 scanpy / pertpy 里都装在一个 `AnnData` 对象里。它有 6 个重要槽位（slot）：

| 槽位 | 存什么 | 形状 |
|---|---|---|
| `.X` | 表达矩阵（核心数据） | 细胞 × 基因 |
| `.obs` | **细胞**的 metadata（每一行 = 一个细胞） | 细胞 × 属性列 |
| `.var` | **基因**的 metadata（每一行 = 一个基因） | 基因 × 属性列 |
| `.layers` | 额外的表达矩阵（如归一化后、log 后） | 细胞 × 基因 |
| `.obsm` | 细胞的低维坐标（如 PCA、UMAP） | 细胞 × 维度 |
| `.uns` | 非结构化的元信息（颜色、参数、图等） | 字典 |

**两条铁律**：

1. `.obs` 和 `.X` 的**行**对齐 —— 都按「细胞」排。
2. `.var` 和 `.X` 的**列**对齐 —— 都按「基因」排。


In [14]:
# 输入：无（只是加载库）
# 输出：无（只是让后续代码可用）
# 目的：引入核心库 —— scanpy（单细胞分析）、pertpy（扰动数据与方法）

import numpy as np
import pandas as pd
import scanpy as sc
import pertpy as pt


In [ ]:
# 输入：无（pertpy 内部会去下载数据）
# 输出：一个 AnnData 对象 adata
# 目的：加载 sci-Plex2 数据
# 注意：第一次运行会下载约 145MB 到 ./data/ 目录，之后直接用缓存

adata = pt.dt.srivatsan_2020_sciplex2()


In [15]:
# 输入：adata
# 输出：两个数字 —— 细胞数、基因数
# 目的：回答「数据有多大」。单细胞里习惯写 (n_obs, n_var) = (细胞, 基因)

print("n_obs（细胞数） =", adata.n_obs)
print("n_vars（基因数） =", adata.n_vars)
print("shape           =", adata.shape)


n_obs（细胞数） = 24262
n_vars（基因数） = 58347
shape           = (24262, 58347)


In [16]:
# 输入：adata.X
# 输出：类型、数据类型、形状、以及一个细胞前几个基因的原始值
# 目的：确认表达矩阵是什么。这里应是「原始 UMI counts」的稀疏矩阵

print("类型  :", type(adata.X).__name__)
print("dtype :", adata.X.dtype)
print("形状  :", adata.X.shape)

# 直接看第 1 个细胞前 5 个基因的原始值（稀疏矩阵要用 toarray() 展开）
print("第 1 个细胞前 5 个基因的值:", adata.X[0, :5].toarray())


类型  : csr_matrix
dtype : float32
形状  : (24262, 58347)
第 1 个细胞前 5 个基因的值: [[0. 0. 0. 0. 0.]]


In [17]:
# 输入：adata.obs
# 输出：21 列的列名 + 每列数据类型
# 目的：看看「每个细胞」身上记录了哪些信息（这是理解 drug/dose/replicate/control 的关键）

print("obs 有", adata.obs.shape[1], "列：")
for col in adata.obs.columns:
    print("   ", col.ljust(24), "->", str(adata.obs[col].dtype))


obs 有 21 列：
    ncounts                  -> int64
    hash_umis                -> float64
    pval_demultiplexing      -> float64
    qval_demultiplexing      -> float64
    top_to_second_best_ratio -> float64
    top_oligo                -> category
    perturbation             -> category
    dose_value               -> category
    well                     -> category
    celltype                 -> category
    cell_line                -> category
    cancer                   -> bool
    disease                  -> category
    tissue_type              -> category
    organism                 -> category
    perturbation_type        -> category
    ngenes                   -> int64
    percent_mito             -> float32
    percent_ribo             -> float32
    nperts                   -> int64
    chembl-ID                -> category


## 逐列解释 `.obs`（细胞 metadata）

把 21 列分成四类：

### ① 扰动实验信息（drug / dose / replicate / control 都在这里）

| 列 | 含义 |
|---|---|
| `perturbation` | 药物名或 "control"。共 5 类：Dex / Nutlin / SAHA / BMS / control |
| `dose_value` | 药物浓度（µM）。8 档：0, 0.1, 0.5, 1, 5, 10, 50, 100 |
| `well` | 孔位编号（共 192 个）。**重复(replicate)信息藏在这里**，没有单独的 replicate 列 |
| `top_oligo` | 该细胞对应的最强 hashtag 寡核苷酸（格式 `药_剂量_孔`） |
| `perturbation_type` | 扰动类型（本数据里全是 "drug"） |
| `nperts` | 该细胞接受了几种扰动（全是 1） |
| `chembl-ID` | 药物 ChEMBL 编号（**只有 Nutlin 填了，其余为空 —— 来源数据标注不全**） |

### ② hashtag 拆分质量控制（怎么把细胞分配到药物）

| 列 | 含义 |
|---|---|
| `hash_umis` | 该细胞 hashtag 寡核苷酸的 UMI 数 |
| `pval_demultiplexing` | 拆分显著性 p 值 |
| `qval_demultiplexing` | 校正后的 q 值（FDR） |
| `top_to_second_best_ratio` | 最强 / 次强寡核苷酸比值（越大越可信） |

### ③ 细胞身份标注

| 列 | 含义 |
|---|---|
| `celltype` | 细胞类型（如 alveolar basal epithelial cells） |
| `cell_line` | 细胞系（A549） |
| `cancer` | 是否癌（True） |
| `disease` | 疾病（lung adenocarcinoma） |
| `tissue_type` | 组织来源（cell_line） |
| `organism` | 物种（human） |

### ④ RNA 质控指标

| 列 | 含义 |
|---|---|
| `ncounts` | 该细胞总 UMI 数（= `.X` 这一行的和，已核实） |
| `ngenes` | 该细胞检测到的基因数 |
| `percent_mito` | 线粒体 reads 百分比 |
| `percent_ribo` | 核糖体 reads 百分比 |

另外：`.obs` 的行索引名是 `cell_barcode`（每个细胞唯一的条形码）。


In [18]:
# 输入：adata.var
# 输出：3 列列名 + 前 3 行
# 目的：看看「每个基因」身上记录了哪些信息

print("var 有", adata.var.shape[1], "列：")
print(adata.var.columns.tolist())
print("\n行索引是基因名，前 3 行：")
adata.var.head(3)


var 有 3 列：
['ensembl_id', 'ncounts', 'ncells']

行索引是基因名，前 3 行：


,ensembl_id,ncounts,ncells
gene_symbol,,,
TSPAN6,ENSG00000000003,4490.0,3426
TNMD,ENSG00000000005,22.0,22
DPM1,ENSG00000000419,13945.0,7532


## 逐列解释 `.var`（基因 metadata）

行索引 = `gene_symbol`（基因名，如 TSPAN6）。

| 列 | 含义 |
|---|---|
| `ensembl_id` | Ensembl 基因 ID（如 ENSG00000000003） |
| `ncounts` | 该基因在所有细胞中的总 UMI 数 |
| `ncells` | 有多少个细胞表达了该基因 |


In [19]:
# 输入：adata
# 输出：三个空列表
# 目的：确认这份数据「还没有」任何预处理产物（归一化层、降维坐标、图等）

print("layers（额外表达矩阵）:", list(adata.layers.keys()))
print("obsm（降维坐标）      :", list(adata.obsm.keys()))
print("uns（非结构化信息）   :", list(adata.uns.keys()))


layers（额外表达矩阵）: []
obsm（降维坐标）      : []
uns（非结构化信息）   : []


In [20]:
# 输入：adata.obs['perturbation']
# 输出：每种药物的细胞数
# 目的：回答「数据里有哪些药物、各有多少细胞」（对应科研问题 1：不同药物）

adata.obs['perturbation'].value_counts()


perturbation
Dex        8064
Nutlin     5956
SAHA       5530
BMS        4183
control     529
Name: count, dtype: int64

In [21]:
# 输入：adata.obs['dose_value']
# 输出：8 个浓度值（从小到大）
# 目的：回答「有哪些剂量」（对应科研问题 2：dose-dependent 的前提）

doses = sorted(adata.obs['dose_value'].dropna().unique(), key=lambda x: float(x))
print(doses)


['0', '0.1', '0.5', '1', '5', '10', '50', '100']


In [22]:
# 输入：adata.obs
# 输出：每个 (药物, 剂量) 组合有多少个 well
# 目的：搞清「重复(replicate)」是怎么记录的 —— 没有专门的 replicate 列，藏在 well 里

n_wells = (
    adata.obs
    .dropna(subset=['dose_value'])
    .groupby(['perturbation', 'dose_value'], observed=True)['well']
    .nunique()
)
print("药物 x 剂量 组合数 =", len(n_wells))
print("每个组合的 well 数（应全部相同）:", n_wells.unique())
print("总 well 数 =", adata.obs['well'].nunique(dropna=True))


药物 x 剂量 组合数 = 32
每个组合的 well 数（应全部相同）: [6]
总 well 数 = 192


In [23]:
# 输入：adata.obs
# 输出：control 细胞的几个特殊之处
# 目的：理解「对照(control)」是怎么标注的

ctrl = adata.obs[adata.obs['perturbation'] == 'control']
print("control 细胞数 =", len(ctrl))
print("control 的 dose_value 是否全为空：", ctrl['dose_value'].isna().all())
print("control 的 well 是否全为空       ：", ctrl['well'].isna().all())
print("control 的 top_oligo 是否全为空  ：", ctrl['top_oligo'].isna().all())


control 细胞数 = 529
control 的 dose_value 是否全为空： True
control 的 well 是否全为空       ： True
control 的 top_oligo 是否全为空  ： True


## 小结：这份数据长什么样

1. **24262 个细胞 × 58347 个基因**，`.X` 是原始 UMI counts（未归一化）。
2. **5 类扰动**：Dex(8064)、Nutlin(5956)、SAHA(5530)、BMS(4183)、control(529)。
3. **8 个剂量**：0（相当于加了 hashtag 的溶剂对照）+ 7 个真实浓度（0.1 ~ 100 µM）。
4. **重复信息在 `well` 里**：每个 (药, 剂量) 组合有 6 个 well，共 192 个 well。
5. **control 细胞没加 hashtag**：所以它的 dose/well/top_oligo 全是空的，靠 `perturbation == 'control'` 识别。
6. `.layers` / `.obsm` / `.uns` 全空 → 数据还是「生」的，等后面步骤再做归一化、降维。

> 要记住的坑：`chembl-ID` 只有 Nutlin 填了值（CHEMBL407632;CHEMBL191334），其它药是空的 —— 不是数据错误，是来源数据标注不全。后续要用药物 ID 时得自己补。


## 检查自己是否真的理解了（不用写代码，用文字回答）

1. 这份数据的 `.X` 是 **(a) 已 log 归一化的表达值** 还是 **(b) 原始 UMI counts**？你凭什么这么说？（提示：看 `ncounts` 和 `.X` 行的关系）

2. 我想查「地塞米松(Dex)在 10 µM 浓度下有多少个细胞」，需要过滤 `.obs` 的哪两列？过滤出来的行和 `.X` 的行是什么关系？

3. `perturbation == 'control'` 的细胞，为什么 `dose_value`、`well`、`top_oligo` 都是空值(NaN)？这暗示这份数据用什么技术把细胞分回它们各自的条件（demultiplexing）？

4. 我说「每个 (药物, 剂量) 组合有 6 个生物学重复」，这个说法严格来说对不对？`well` 到底代表「生物学重复」还是「技术/孔重复」，单凭这份数据能区分吗？

5. 做「不同药物 → 不同转录状态」（科研问题 1）时，用 `perturbation` 分组会天然漏掉哪类细胞？另外，为什么 high-dose 组的细胞数普遍比 low-dose 少？（提示：把剂量和细胞数放一起看）
